# TextCNN — 법률 판례 사건 유형 분류
팀 레옹 | 학습형 임베딩 기반 딥러닝 비교 모델

**역할**: TF-IDF(희소 빈도)와 BERT(사전학습 문맥) 사이의 **학습 임베딩** 비교군.
TF-IDF처럼 n-gram 패턴을 보지만, 빈도가 아니라 데이터로부터 직접 학습한다.

**공통 규칙 (다른 노트북과 통일)**
- 입력: 결합 (`jdgmn + ' ' + summ_pass`)
- 데이터 경로: `../processed_data/`
- 결과 CSV 컬럼: `[모델, 입력, Accuracy, F1-macro, Precision, Recall]`
- 라벨: `label_mapping.csv` 기준 (0=가사 ... 9=형사B)


## 1단계: 환경 설정 (RunPod에서 1회만)
한국어 형태소 분석기(KoNLPy/Okt)는 Java가 필요하다.

In [ ]:
# RunPod / Colab 첫 실행 시에만 (로컬에 이미 깔려있으면 건너뛰기)
# !apt-get install -y default-jdk > /dev/null 2>&1
# !pip install konlpy torch scikit-learn pandas matplotlib seaborn
print("환경 설정 셀 — 필요 시 위 주석 해제 후 실행")

## 2단계: 라이브러리 로드 및 시드 고정
재현성을 위해 랜덤 시드를 모두 고정한다.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from konlpy.tag import Okt

# 시드 고정 (결과 재현용)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# GPU 사용 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 장치: {device}")
print("라이브러리 로드 완료")

## 3단계: 데이터 로드
baseline과 동일하게 `../processed_data/`에서 불러온다.

In [ ]:
# 데이터 로드
train = pd.read_csv('../processed_data/train.csv')
val   = pd.read_csv('../processed_data/val.csv')
test  = pd.read_csv('../processed_data/test.csv')
label_map = pd.read_csv('../processed_data/label_mapping.csv')

# 라벨 이름 (숫자 -> 카테고리명), 숫자 순서대로 정렬
label_names = label_map.sort_values('label')['class_name'].tolist()
num_classes = len(label_names)

# 입력 텍스트 = 결합 (판시사항 + 요약문). 결측치는 빈 문자열로.
def make_input(df):
    return (df['jdgmn'].fillna('') + ' ' + df['summ_pass'].fillna('')).tolist()

train_texts = make_input(train)
val_texts   = make_input(val)
test_texts  = make_input(test)

train_labels = train['label'].tolist()
val_labels   = val['label'].tolist()
test_labels  = test['label'].tolist()

print(f"Train: {len(train_texts)}건 | Val: {len(val_texts)}건 | Test: {len(test_texts)}건")
print(f"클래스 {num_classes}개: {label_names}")

## 4단계: 한국어 형태소 분석 (Okt)
한국어는 띄어쓰기만으로는 의미 단위가 안 나오므로 형태소 분석기로 쪼갠다.
**주의**: 34,000건 형태소 분석은 5~15분 걸린다. 한 번 돌리고 결과를 캐싱하면 좋다.

In [ ]:
okt = Okt()

# 형태소 단위로 토큰화하는 함수 (명사/동사/형용사 등 의미있는 품사 위주)
def tokenize(text):
    # morphs: 형태소 단위로 쪼갬. stem=True로 동사/형용사 원형 통일
    return okt.morphs(text, stem=True)

# 시간 오래 걸리므로 캐싱 (이미 만들어둔 게 있으면 로드)
CACHE = 'tokens_cache.npz'
if os.path.exists(CACHE):
    print("캐시 로드 중...")
    cache = np.load(CACHE, allow_pickle=True)
    train_tokens = cache['train'].tolist()
    val_tokens   = cache['val'].tolist()
    test_tokens  = cache['test'].tolist()
else:
    print("형태소 분석 시작 (5~15분 소요)...")
    train_tokens = [tokenize(t) for t in train_texts]
    val_tokens   = [tokenize(t) for t in val_texts]
    test_tokens  = [tokenize(t) for t in test_texts]
    # 캐싱 저장 (다음 실행부터 빠름)
    np.savez(CACHE,
             train=np.array(train_tokens, dtype=object),
             val=np.array(val_tokens, dtype=object),
             test=np.array(test_tokens, dtype=object))
    print("토큰 캐싱 완료 -> tokens_cache.npz")

print(f"토큰화 예시: {train_tokens[0][:15]}")

## 5단계: 단어 사전(vocab) 구축
단어를 숫자로 바꾸기 위한 사전을 만든다. **학습 데이터로만** 만든다 (정보 누수 방지).
- `0` = PAD (길이 맞추는 빈칸)
- `1` = UNK (사전에 없는 단어)

In [ ]:
from collections import Counter

# 학습 데이터의 단어 빈도 계산
counter = Counter()
for tokens in train_tokens:
    counter.update(tokens)

# 너무 드문 단어는 제외 (2번 이상 등장한 단어만), 최대 30000개
MAX_VOCAB = 30000
MIN_FREQ = 2

vocab = {'<PAD>': 0, '<UNK>': 1}
for word, freq in counter.most_common(MAX_VOCAB):
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

vocab_size = len(vocab)
print(f"단어 사전 크기: {vocab_size}개")

# 토큰 리스트 -> 숫자 ID 리스트로 변환하는 함수
def encode(tokens):
    return [vocab.get(w, 1) for w in tokens]  # 없으면 UNK(1)

## 6단계: Dataset / DataLoader 정의
문장 길이를 `MAX_LEN`으로 통일한다 (긴 건 자르고, 짧은 건 PAD로 채움).
판시사항이 길어서 256 정도면 충분 (EDA상 평균 토큰 ~175).

In [ ]:
MAX_LEN = 256  # 문장 최대 길이 (토큰 수)

# 숫자 ID 리스트를 MAX_LEN 길이로 맞추는 함수
def pad_sequence(ids):
    if len(ids) >= MAX_LEN:
        return ids[:MAX_LEN]          # 길면 자르기
    return ids + [0] * (MAX_LEN - len(ids))  # 짧으면 PAD(0)로 채우기

class TextDataset(Dataset):
    def __init__(self, token_lists, labels):
        # 미리 인코딩 + 패딩해서 텐서로 보관
        self.X = torch.tensor(
            [pad_sequence(encode(t)) for t in token_lists], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# DataLoader 생성
BATCH_SIZE = 64
train_ds = TextDataset(train_tokens, train_labels)
val_ds   = TextDataset(val_tokens,   val_labels)
test_ds  = TextDataset(test_tokens,  test_labels)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print(f"DataLoader 준비 완료 (batch={BATCH_SIZE}, max_len={MAX_LEN})")

## 7단계: TextCNN 모델 정의
여러 크기의 필터(kernel)로 n-gram 패턴을 잡는다.
- kernel=3 -> 3단어 패턴, kernel=4 -> 4단어, kernel=5 -> 5단어
- 각 필터의 결과를 max-pooling으로 요약 후 합쳐서 분류

In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_classes=10,
                 kernel_sizes=(3, 4, 5), num_filters=100, dropout=0.5):
        super().__init__()
        # 단어 임베딩 (PAD=0은 학습 안 함)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # 서로 다른 크기의 1D 합성곱 필터들
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        # 필터 결과를 모두 합친 뒤 분류
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embedding(x)              # (batch, seq_len, embed_dim)
        x = x.transpose(1, 2)              # (batch, embed_dim, seq_len) <- Conv1d 형식
        # 각 필터 통과 -> ReLU -> 시간축 max pooling
        pooled = []
        for conv in self.convs:
            c = torch.relu(conv(x))        # (batch, num_filters, L')
            p = torch.max(c, dim=2)[0]     # (batch, num_filters)
            pooled.append(p)
        out = torch.cat(pooled, dim=1)     # 필터 결과 합치기
        out = self.dropout(out)
        return self.fc(out)

model = TextCNN(vocab_size=vocab_size, num_classes=num_classes).to(device)
print(model)

## 8단계: 손실함수 / 옵티마이저
클래스 불균형(56배)이 심하므로 **class_weight**를 손실함수에 적용한다.
baseline의 `class_weight='balanced'`와 같은 효과.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# 클래스별 가중치 계산 (적은 클래스에 더 큰 가중치)
classes = np.arange(num_classes)
weights = compute_class_weight('balanced', classes=classes, y=np.array(train_labels))
class_weights = torch.tensor(weights, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("클래스 가중치:", np.round(weights, 2))

## 9단계: 학습 (Validation F1 기준 best 저장)
각 epoch마다 검증셋 F1-macro를 측정하고, 가장 좋은 모델을 저장한다 (과적합 방지).

In [ ]:
# 검증/평가용 예측 함수
def predict(loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            logits = model(X)
            preds.extend(logits.argmax(dim=1).cpu().numpy())
            trues.extend(y.numpy())
    return np.array(preds), np.array(trues)

EPOCHS = 10
best_f1 = 0.0
best_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # 검증셋 F1 측정
    val_pred, val_true = predict(val_loader)
    val_f1 = f1_score(val_true, val_pred, average='macro')
    print(f"[Epoch {epoch:2d}] loss={total_loss/len(train_loader):.4f} | val F1-macro={val_f1:.4f}")

    # best 모델 저장
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\n최고 검증 F1-macro: {best_f1:.4f}")
# 가장 좋았던 가중치 복원
model.load_state_dict(best_state)

## 10단계: Test 평가 + 결과 저장
baseline과 **동일한 CSV 형식**으로 저장 -> 나중에 합칠 때 안 깨짐.

In [ ]:
# Test 평가
test_pred, test_true = predict(test_loader)

print("=== TextCNN (결합) Test 결과 ===")
print(classification_report(test_true, test_pred, target_names=label_names))

report = classification_report(test_true, test_pred, target_names=label_names, output_dict=True)

# baseline_results.csv와 동일한 컬럼 형식
result = {
    '모델': 'TextCNN',
    '입력': '결합',
    'Accuracy':  round(accuracy_score(test_true, test_pred), 4),
    'F1-macro':  round(report['macro avg']['f1-score'], 4),
    'Precision': round(report['macro avg']['precision'], 4),
    'Recall':    round(report['macro avg']['recall'], 4),
}
result_df = pd.DataFrame([result])
print(result_df.to_string(index=False))

# CSV 저장
result_df.to_csv('textcnn_results.csv', index=False)
print("\ntextcnn_results.csv 저장 완료")

## 11단계: Confusion Matrix
baseline과 같은 형식으로 시각화 (행별 정규화).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 (RunPod 환경: NanumGothic, 로컬 mac: AppleGothic)
for font in ['NanumGothic', 'AppleGothic', 'Malgun Gothic']:
    try:
        matplotlib.rcParams['font.family'] = font
        break
    except Exception:
        continue
matplotlib.rcParams['axes.unicode_minus'] = False

cm = confusion_matrix(test_true, test_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # 행별 정규화

plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, xticklabels=label_names, yticklabels=label_names,
            annot=True, fmt='.2f', cmap='Blues')
plt.title('TextCNN Confusion Matrix (결합, 행별 정규화)', fontsize=14)
plt.xlabel('예측')
plt.ylabel('실제')
plt.tight_layout()
plt.savefig('confusion_matrix_textcnn.png', dpi=150)
plt.show()
print("confusion_matrix_textcnn.png 저장 완료")

## 12단계 (선택): Late Fusion용 확률 저장
하이브리드(Late Fusion)에서 TextCNN 확률도 후보로 쓸 수 있게 저장해둔다.
김홍근의 BERT 확률과 같은 형식(test 샘플 x 10클래스)으로 맞춤.

In [ ]:
# Test 확률(softmax) 추출
model.eval()
all_probs = []
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        probs = torch.softmax(model(X), dim=1)
        all_probs.append(probs.cpu().numpy())
test_probs = np.concatenate(all_probs, axis=0)  # (test건수, 10)

np.save('textcnn_test_probs.npy', test_probs)
print(f"textcnn_test_probs.npy 저장 완료 — shape={test_probs.shape}")